In [1]:
# import newkernelo as lib
import xllim as lib
import kernelo as oldker
import numpy as np
import time
import os.path
import json
import logging
logging.getLogger().setLevel(logging.INFO)

In [ ]:
K = 50
D = 9
L = 4

# ! Hybrid model
n_hidden_variables = 0
L_t = L - n_hidden_variables
L_w = n_hidden_variables

gllim = lib.GLLiM(K, D, L, gamma_type='full', sigma_type='diag', n_hidden_variables=n_hidden_variables)
print(type(gllim))
gllim.getDimensions()
gllim.getConstraints()
print(gllim.getParamPi())
gllim.getParamGamma()
print(gllim.getParamGamma().shape)


In [ ]:
lib._GLLiM_full_diag(K, D, L, 'full', 'full', n_hidden_variables)

In [ ]:
print(gllim.getDimensions())
gllim.getParams()
gllim.getParamGamma().shape

In [ ]:
theta = gllim.getParams()
print(theta.Pi)
print(theta.A.shape)
# print(theta.L)
print(gllim.getDimensions())

In [ ]:
gllim.getParams()
gllim.getParamPi()
gllim.getParamA()
gllim.getParamC()
gllim.getParamGamma()
gllim.getParamB()
gllim.getParamSigma()
print(gllim.getDimensions())

In [ ]:
gllim.getInverse().C
print(gllim.getDimensions())

In [ ]:
Pi = np.ones(K) * 1 / K
A = np.ones((K, D, L))
C = np.ones((K, L)) * 2
Gamma = np.ones((K, L, L)) * 0.2
B = np.ones((K, D)) * 3
Sigma = np.ones((K, D, D)) * 0.3
print(Gamma.shape)
gllim.setParamPi(Pi)
gllim.setParamA(A)
gllim.setParamC(C)
gllim.setParamGamma(Gamma)
gllim.setParamB(B)
gllim.setParamSigma(np.array(Sigma[:, :, 0]))

theta = gllim.getParams()
print(theta.Pi)
print(theta.A)
print(theta.C)
print(theta.Gamma)
print(theta.B)
print(theta.Sigma)
print(theta.Sigma.shape)
print(gllim.getDimensions())

In [ ]:
print(gllim.getDimensions())
gllim.getParams()
print(gllim.getDimensions())

In [ ]:
new_theta = lib.GLLiMParameters(K,D,L, "full", "diag")
print(type(new_theta))

Gamma = np.ones((K,L,L)) + np.random.rand(K,L,L) * 0.1
Sigma = np.ones((K,D)) + np.random.rand(K,D) * 0.1
for k in range(K):
    Gamma[k] += np.eye(L) * 0.1
    Gamma[k] = np.dot(Gamma[k], Gamma[k].T) * 0.001
    # Sigma[k] += np.eye(D) * 0.1
    # Sigma[k] = np.dot(Sigma[k], Sigma[k].T) * 0.001
new_theta.Pi = Pi
new_theta.A = A * 3
new_theta.C = C * 2
new_theta.Gamma = Gamma
new_theta.B = B * 2
new_theta.Sigma = Sigma
print(gllim.getDimensions())
gllim.setParams(new_theta)
print(gllim.getDimensions()) # TODO ERROR GLLiM<TGamma,TSigma> dimensions are (L=4000838736, D=32766, K=4000837888)
theta = gllim.getParams()
print(theta.Pi)
print(theta.A)
print(theta.C)
print(theta.Gamma)
print(theta.B)
print(theta.Sigma)

In [ ]:
theta_star = gllim.getInverse()

print(theta_star.Pi)
print(theta_star.A)
print(theta_star.C)
print(theta_star.Gamma)
print(theta_star.B)
print(theta_star.Sigma)

In [ ]:
x = np.random.rand(L, 6)
x_incertitudes = np.random.rand(L) * 0.2
print(x)
prediction_direct_results = gllim.directDensities(x, x_incertitudes)

In [ ]:
print(prediction_direct_results.fullGMM.mean)
print(prediction_direct_results.fullGMM.variance)
print(prediction_direct_results.fullGMM.weights)
print(prediction_direct_results.fullGMM.means)

In [ ]:
y = np.random.rand(D, 40)
y_incertitudes = np.random.rand(D,1) * 0.01
# print(y)
# print(y_incertitudes)
tic = time.time()
prediction_results = gllim.inverseDensities(y, y_incertitudes)
print("Time One inversion = {}".format(time.time()-tic))

In [15]:
# print(prediction_results.fullGMM.mean)
# print(prediction_results.fullGMM.variance)
# print(prediction_results.fullGMM.weights)
# print(prediction_results.fullGMM.means)

In [ ]:
import numpy.matlib
y_incertitudes_temp = np.matlib.repmat(y_incertitudes,1,40)
y_incertitudes_mat = np.copy(y_incertitudes_temp)
# y_incertitudes_mat = np.random.rand(D, 4) * 0.01
# print(y_incertitudes_mat)
tic = time.time()
prediction_results_all = gllim.inverseDensities(y, y_incertitudes_mat)
print("Time Multi inversion = {}".format(time.time()-tic))

#### Prediction with merging gaussians algorithm

In [ ]:
K_merged = 2
merging_threshold = 1e-5

tic = time.time()
prediction_results_all = gllim.inverseDensities(y, y_incertitudes_mat, K_merged, merging_threshold, 2)
print("Time Multi inversion = {}".format(time.time()-tic))

In [ ]:
print(prediction_results_all.fullGMM.weights.shape)
print(prediction_results_all.mergedGMM.weights.shape)
print(prediction_results_all.mergedGMM.means.shape)
print(len(prediction_results_all.mergedGMM.covs))
print(prediction_results_all.mergedGMM.covs[0].shape)

print("sum wieghts for pred by mean and for centers:")
print(np.sum(prediction_results_all.fullGMM.weights[0]))
print(np.sum(prediction_results_all.mergedGMM.weights[0]))

print("weights & Centers:")
print(prediction_results_all.mergedGMM.weights[0])
print(prediction_results_all.mergedGMM.means[0])

print("pred by mean:")
print(prediction_results_all.fullGMM.mean[0])

print("mean of centers:")
mean_of_centers = prediction_results_all.mergedGMM.weights[0,0] * prediction_results_all.mergedGMM.means[0,:,0]\
                + prediction_results_all.mergedGMM.weights[0,1] * prediction_results_all.mergedGMM.means[0,:,1]
print(mean_of_centers)
print(prediction_results_all.mergedGMM.mean[0])

In [ ]:
print("\nmean")
print(prediction_results_all.fullGMM.mean)
print(prediction_results.fullGMM.mean)
print(np.all(np.isclose(prediction_results.fullGMM.mean, prediction_results_all.fullGMM.mean)))
# print("\nvariance")
# print(prediction_results_all.fullGMM.variance)
# print(prediction_results_all.fullGMM.weights)
# print(prediction_results_all.fullGMM.means)

#### gllim.regularize(series)

In [ ]:
series = prediction_results_all.mergedGMM.means
permutations = gllim.regularize(series)
print(permutations.shape)
print(permutations)

# GLLiM TRAIN

In [21]:
# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
# physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", dir_path + "/pytest/models").create()
physical_model = oldker.TestModelConfig().create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Create GLLIM model, including its initialization and training configuration
learningConfig = oldker.EMLearningConfig(10, 1e-5, 1e-12)
# learningConfig=oldker.GMMLearningConfig(15, 100, 1e-12)
# initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=1, gmmLearningConfig=oldker.GMMLearningConfig(0, 0, 1e-12))
gllim_oldker= oldker.GLLiM(physical_model.get_D_dimension(), physical_model.get_L_dimension(), 50, "Full", "Diag", initConfig, learningConfig)


In [ ]:
# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(1000)
print("Initializing GLLIM model")
gllim_oldker.initialize(x_gen, y_gen)
gllim_parameters_0 = gllim_oldker.exportModel() # you can export your gllim model parameters
print("Training model")
gllim_oldker.train(x_gen, y_gen)

gllim_parameters = gllim_oldker.exportModel() # you can export your gllim model parameters


### Get theta_0 from oldker

In [ ]:
theta_0 = lib.GLLiMParameters(K,D,L, "full", "diag")

print(np.array(gllim_parameters_0.Pi).shape)
print(np.array(gllim_parameters_0.A).shape)
print(np.array(gllim_parameters_0.C).shape)
print(np.array(gllim_parameters_0.Gamma).shape)
print(np.array(gllim_parameters_0.B).shape)
print(np.array(gllim_parameters_0.Sigma).shape)

# print(np.array(gllim_parameters_0.Pi))
# print(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)).shape)

theta_0.Pi = np.copy(np.array(gllim_parameters_0.Pi))
theta_0.A = np.copy(np.array(gllim_parameters_0.A))
theta_0.C = np.copy(np.array(gllim_parameters_0.C.T))
# theta_0.Gamma = np.copy(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)))
theta_0.Gamma = np.copy(gllim_parameters_0.Gamma)
theta_0.B = np.copy(np.array(gllim_parameters_0.B.T))
# theta_0.Sigma = np.copy(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)))
Sigma_diag = np.zeros((K,D))
for k in range(K):
    Sigma_diag[k,:] = np.diag(gllim_parameters_0.Sigma[k])
theta_0.Sigma = Sigma_diag

print(np.sum(theta_0.Pi))
gllim.setParams(theta_0)

In [24]:
# theta_0 = theta
# print(theta_0.Pi)
# print(theta_0.C)

# jgmm = lib.JGMM()
# theta_jgmm = jgmm.train(x_gen, y_gen, theta_0, 15, 10, 1e-12)
# print(theta_0.Pi)
# print(theta_jgmm.Pi)
# print(theta_0.C)
# print(theta_jgmm.C)

# TODO ça plante parceque le V (dans jgmm.cpp) n'est pas sym strict def positif. Surement que dans gllim_parameters_0 a un Gamma et Sigma qui n'est pas sym strict def positif

In [25]:
# # print(theta.Pi)
# gllim.setParams(theta)


# # print(gllim.getParamPi())
# # print(gllim.getParamC())
# gllim.train(x_gen, y_gen, 15, 100, 1e-12)
# time.sleep(1)
# print(gllim.getParamPi()) 
# print(gllim.getParamC())

# # TODO le gllim.getParamPi() doit changer !


In [26]:
# print(gllim_parameters.Pi)
# print(theta_jgmm.Pi)
# print(gllim.getParamPi())

## GLLM training : EM estimator : full/full

In [27]:
L, D, K, N = 6, 30, 5, 100
theta_full = lib.GLLiMParameters(K, D, L, "full", "full")

Gamma = np.random.rand(K, L, L)
Sigma = np.random.rand(K, D, D)
for k in range(K):
    Gamma[k, :, :] += np.eye(L) * 0.1
    Gamma[k, :, :] = np.dot(Gamma[k, :, :], Gamma[k, :, :].T) * 0.001
    Sigma[k, :, :] += np.eye(D) * 0.1
    Sigma[k, :, :] = np.dot(Sigma[k, :, :], Sigma[k, :, :].T) * 0.001
theta_full.Pi = np.ones(K) * 1 / K
theta_full.A = np.ones((K, D, L)) * 0.3
theta_full.C = np.ones((K, L)) * 0.2
theta_full.Gamma = Gamma
theta_full.B = np.ones((K, D)) * 0.5
theta_full.Sigma = Sigma

X_gen = np.random.rand(
    L, N
)  # TODO transpose X,Y in the python bindings in order to use the transposed version in Python (row-major)
Y_gen = np.random.rand(D, N)

In [28]:
L_t = L
full_full_gllim = lib.GLLiM(K, D, L, 'full', 'full', n_hidden_variables)
full_full_gllim.setParams(theta_full)

In [ ]:
print(theta_full.Pi)
print(full_full_gllim.getParamPi())
print(full_full_gllim.getParamGamma())
print(theta_full.Gamma)

In [ ]:
print(full_full_gllim.getParamPi())
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 12
gmm_floor = 1e-12
nb_experiences = 3
seed = 12345
verbose = 1
full_full_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
time.sleep(3)
print(full_full_gllim.getParamPi())

In [ ]:
print(full_full_gllim.getParamB())
# x_gen_T = np.array(x_gen.T)
# y_gen_T = np.array(y_gen.T)
full_full_gllim.train(X_gen, Y_gen, 30, 1e-3, 1e-12, 0)
print(full_full_gllim.getParamB())
# print(full_full_gllim.getParamSigma())

#### Initialize + train GLLiM full/full from scratch

In [ ]:
new_full_full_gllim = lib.GLLiM(K, D, L, 'full', 'full', n_hidden_variables)
print(new_full_full_gllim.getParamPi())
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 12
gmm_floor = 1e-12
nb_experiences = 3
seed = 12345
verbose = 1
new_full_full_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
time.sleep(1)
print(new_full_full_gllim.getParamPi())

print(new_full_full_gllim.getParamPi())
new_full_full_gllim.train(X_gen, Y_gen, 30, 1e-3, 1e-12, verbose)
time.sleep(1)
print(new_full_full_gllim.getParamPi())

## GLLM training : EM estimator : full/diag

In [33]:
L, D, K, N = 6, 30, 5, 4
new_theta = lib.GLLiMParameters(K, D, L, "full", "diag")

Gamma = np.random.rand(K, L, L)
Sigma = np.ones((K, D)) + np.random.rand(K, D) * 0.1
for k in range(K):
    Gamma[k, :, :] += np.eye(L) * 0.1
    Gamma[k, :, :] = np.dot(Gamma[k, :, :], Gamma[k, :, :].T) * 0.001
    # Sigma[k,:,:] += np.eye(D) * 0.1
    # Sigma[k,:,:] = np.dot(Sigma[k,:,:], Sigma[k,:,:].T) * 0.001
new_theta.Pi = np.ones(K) * 1 / K
new_theta.A = np.random.rand(K, D, L) * 0.3
new_theta.C = np.random.rand(K, L) * 0.2
new_theta.Gamma = Gamma
new_theta.B = np.random.rand(K, D) * 0.5
new_theta.Sigma = Sigma

X_gen = np.random.rand(L, N)
Y_gen = np.random.rand(D, N)

In [34]:
diag_gllim = lib.GLLiM(K, D, L, 'full', 'diag', n_hidden_variables)
diag_gllim.setParams(new_theta)

In [ ]:
print(new_theta.Pi)
print(diag_gllim.getParamPi())
print(diag_gllim.getParamGamma())
print(new_theta.Gamma)

In [ ]:
print(diag_gllim.getParamPi())
# x_gen_T = np.array(x_gen.T)
# y_gen_T = np.array(y_gen.T)
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 12
gmm_floor = 1e-12
nb_experiences = 3
seed = 12345
verbose = 1
diag_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
time.sleep(3)
print(diag_gllim.getParamPi())

In [ ]:
print(diag_gllim.getParamPi())
# x_gen_T = np.array(x_gen.T)
# y_gen_T = np.array(y_gen.T)
diag_gllim.train(X_gen, Y_gen, 30, 1e-3, 1e-12, 0)
print(diag_gllim.getParamPi())

#### Initialize + train GLLiM full/diag from scratch

In [ ]:
new_full_diag_gllim = lib.GLLiM(K, D, L, 'full', 'diag', n_hidden_variables)
print(new_full_diag_gllim.getParamPi())
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 12
gmm_floor = 1e-12
nb_experiences = 1
seed = 12345
verbose = 1
new_full_diag_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
time.sleep(1)
print(new_full_diag_gllim.getParamPi())

print(new_full_diag_gllim.getParamPi())
new_full_diag_gllim.train(X_gen, Y_gen, 30, 1e-3, 1e-12, verbose)
time.sleep(1)
print(new_full_diag_gllim.getParamPi())

In [ ]:
import os
import xllim as lib
import numpy as np
import time
import logging
logging.getLogger().setLevel(logging.INFO)
L, D, K, N = 6, 30, 50, 4
X_gen = np.random.rand(L,N)
Y_gen = np.random.rand(D,N)

new_full_diag_gllim = lib.GLLiM(K, D, L, 'full', 'diag')
print(new_full_diag_gllim.getParamPi())
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 12
gmm_floor = 1e-12
nb_experiences = 1
seed = 12345
verbose = 1
new_full_diag_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
time.sleep(1)
print(new_full_diag_gllim.getParamPi())

print(new_full_diag_gllim.getParamPi())
new_full_diag_gllim.train(X_gen, Y_gen, 100, -1e-3, 1e-12, verbose)
time.sleep(1)
print(new_full_diag_gllim.getParamPi())

In [ ]:
import os
os.system("")

print("\033[33mtest")
print("\033[33mtest")
print("\033[33m test\rTEST")
print("\033[34m  BLABLABLA")
print("\033[34m  BLA\033[35mBLABLA \r                                        \r\033[2Km blabla")
print("YOU CAN'T SEE ME\033[2K\rhello")